#Section B

##B1 Chunking and Overlapping

I used a chunk size of 250 characters so that each chunk contains a reasonable amount of information. And i used an experimental overlap of 45 characters to preserve some context between consecutive chunks when information is split across chunks.

In [1]:
corpus = """
So first of all Ryan Gosling is literally me, Ryan Gosling is a Canadian actor known for taking on a wide variety of roles across different genres.
His film career includes romantic dramas, crime thrillers, science fiction films, comedies, and musicals.
Some of his well-known movies include The Notebook, Drive, La La Land, Blade Runner 2049, and Barbie.

The Notebook is a romantic drama released in 2004, in which Ryan Gosling plays Noah Calhoun.
The story follows Noah and Allie, two people who fall deeply in love but face opposition from Allie's family.
The movie became one of Gosling's most recognizable romantic roles and helped establish him as a leading actor.

Drive is a crime thriller released in 2011 where Gosling plays an unnamed character simply known as the Driver.
The Driver works as a mechanic and stunt driver while also doing jobs as a getaway driver.
The movie is known for its stylish visuals, atmospheric soundtrack, and relatively quiet lead performance.

La La Land is a musical romantic comedy-drama released in 2016.
Gosling plays Sebastian Wilder, a jazz musician who dreams of opening his own jazz club.
The film follows Sebastian and aspiring actress Mia, played by Emma Stone, as they pursue their careers while developing a relationship.

Blade Runner 2049 is a science fiction film released in 2017 and directed by Denis Villeneuve.
Gosling plays Officer K, a replicant working as a blade runner who discovers information that leads him to investigate a long-hidden mystery.
The movie explores themes such as identity, memory, artificial intelligence, and what it means to be human.

Barbie was released in 2023 and features Gosling as Ken opposite Margot Robbie as Barbie.
The film follows Barbie as she leaves Barbieland and enters the real world, while Ken also begins to question his role and identity.
Gosling's performance as Ken combines comedy, music, and a character who is trying to understand his place in the world.

The Fall Guy is an action comedy released in 2024, with Gosling playing Colt Seavers.
Colt is a stuntman who is brought back into the world of filmmaking while investigating the disappearance of an actor.
The movie combines action, comedy, romance, and the behind-the-scenes world of Hollywood filmmaking.
"""

In [15]:
def chunk_corpus(text, chunk_size, overlap):
  chunks = []

  start = 0
  step = chunk_size - overlap

  while start < len(text):
    end = start + chunk_size
    chunk = text[start:end]
    chunks.append(chunk)

    start += step

  return chunks


chunk_size = 250
overlap = 45

'''
so here like basically the first chunk will start from index 0 to index 250,
but the second chunk instead of starting from index 251 will start from index[250 - 45]
this will help in preserving the context
'''

chunks = chunk_corpus(corpus, chunk_size, overlap)

print(f"Total Chunks formed: {len(chunks)}")

for i,chunk in enumerate(chunks):
  print(f"\nChunk {i+1}:\n{chunk} ")


Total Chunks formed: 12

Chunk 1:

So first of all Ryan Gosling is literally me, Ryan Gosling is a Canadian actor known for taking on a wide variety of roles across different genres. 
His film career includes romantic dramas, crime thrillers, science fiction films, comedies, and musi 

Chunk 2:
rs, science fiction films, comedies, and musicals. 
Some of his well-known movies include The Notebook, Drive, La La Land, Blade Runner 2049, and Barbie.

The Notebook is a romantic drama released in 2004, in which Ryan Gosling plays Noah Calhoun. 
T 

Chunk 3:
 in which Ryan Gosling plays Noah Calhoun. 
The story follows Noah and Allie, two people who fall deeply in love but face opposition from Allie's family. 
The movie became one of Gosling's most recognizable romantic roles and helped establish him as  

Chunk 4:
e romantic roles and helped establish him as a leading actor.

Drive is a crime thriller released in 2011 where Gosling plays an unnamed character simply known as the Driver. 
The 

##B2 Chunk Embeddings



In [16]:
!pip install sentence-transformers

In [18]:
from sentence_transformers import SentenceTransformer
import numpy as np

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

chunk_embeds = embedding_model.encode(chunks)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [19]:
print(f"Number of embeddings: {len(chunk_embeds)}")
print(f"Embedding shape: {chunk_embeds.shape}")

Number of embeddings: 12
Embedding shape: (12, 384)


In [20]:
def calculate_cosine_similarity(a,b):
  return np.dot(a,b)/(np.linalg.norm(a) * np.linalg.norm(b))

In [22]:
def retrieve(query, chunks, chunk_embeds, k=3):

  similarities = []

  query_embed = embedding_model.encode(query)

  for i, chunk_embed in enumerate(chunk_embeds):
    score = calculate_cosine_similarity(query_embed, chunk_embed)

    similarities.append((i, score))

  similarities.sort(key= lambda x: x[1], reverse= True)

  top_results = similarities[:k]

  return top_results

so this is the dense retrieval, instead of looking for the exact answer, this method will look for the closest answer based completely on the vector distance and it will give the following problem.

In [45]:
query1 = "Am I ryan gosling ?"

result1 = retrieve(query1, chunks, chunk_embeds)

print("Query: ", query1)

for rank, (i, score) in enumerate(result1, start = 1):
  print(f"Rank {rank}")
  print(f"Similarity with the query: {score:.4f}")
  print(f"Chunk: {chunks[i]}")

#In this case it worked

Query:  Am I ryan gosling ?
Rank 1
Similarity with the query: 0.7144
Chunk: 
So first of all Ryan Gosling is literally me, Ryan Gosling is a Canadian actor known for taking on a wide variety of roles across different genres. 
His film career includes romantic dramas, crime thrillers, science fiction films, comedies, and musi
Rank 2
Similarity with the query: 0.5708
Chunk:  in which Ryan Gosling plays Noah Calhoun. 
The story follows Noah and Allie, two people who fall deeply in love but face opposition from Allie's family. 
The movie became one of Gosling's most recognizable romantic roles and helped establish him as 
Rank 3
Similarity with the query: 0.5358
Chunk: rs, science fiction films, comedies, and musicals. 
Some of his well-known movies include The Notebook, Drive, La La Land, Blade Runner 2049, and Barbie.

The Notebook is a romantic drama released in 2004, in which Ryan Gosling plays Noah Calhoun. 
T


But in the below query even though the information is present in the corpus this method is not able to retrieve it because in the corpus it is "features Gosling as Ken opposite Margot Robbie as Barbie" and in in our query we are asking for "Ryan Gosling".

In [39]:
query2 = "In which movie was Ryan Gosling paired with Margot Robbie ?"

result2 = retrieve(query2, chunks, chunk_embeds)

print("Query: ", query2)

for rank, (i, score) in enumerate(result2, start = 1):
  print(f"Rank {rank}")
  print(f"Similarity with the query: {score:.4f}")
  print(f"Chunk: {chunks[i]}")

Query:  In which movie was Ryan Gosling paired with Margot Robbie ?
Rank 1
Similarity with the query: 0.5838
Chunk:  in which Ryan Gosling plays Noah Calhoun. 
The story follows Noah and Allie, two people who fall deeply in love but face opposition from Allie's family. 
The movie became one of Gosling's most recognizable romantic roles and helped establish him as 
Rank 2
Similarity with the query: 0.5426
Chunk: 
So first of all Ryan Gosling is literally me, Ryan Gosling is a Canadian actor known for taking on a wide variety of roles across different genres. 
His film career includes romantic dramas, crime thrillers, science fiction films, comedies, and musi
Rank 3
Similarity with the query: 0.5375
Chunk: rs, science fiction films, comedies, and musicals. 
Some of his well-known movies include The Notebook, Drive, La La Land, Blade Runner 2049, and Barbie.

The Notebook is a romantic drama released in 2004, in which Ryan Gosling plays Noah Calhoun. 
T


See here , now when we reform the query and ask just "Gosling", only then its able to give the correct answer.

In [38]:
query3 = "In which movie was Gosling paired with Margot Robbie ?"

result3 = retrieve(query3, chunks, chunk_embeds)

print("Query: ", query3)

for rank, (i, score) in enumerate(result3, start = 1):
  print(f"Rank {rank}")
  print(f"Similarity with the query: {score:.4f}")
  print(f"Chunk: {chunks[i]}")

Query:  In which movie was Gosling paired with Margot Robbie ?
Rank 1
Similarity with the query: 0.5593
Chunk: as released in 2023 and features Gosling as Ken opposite Margot Robbie as Barbie. 
The film follows Barbie as she leaves Barbieland and enters the real world, while Ken also begins to question his role and identity. 
Gosling's performance as Ken comb
Rank 2
Similarity with the query: 0.4973
Chunk:  in which Ryan Gosling plays Noah Calhoun. 
The story follows Noah and Allie, two people who fall deeply in love but face opposition from Allie's family. 
The movie became one of Gosling's most recognizable romantic roles and helped establish him as 
Rank 3
Similarity with the query: 0.4660
Chunk: rs, science fiction films, comedies, and musicals. 
Some of his well-known movies include The Notebook, Drive, La La Land, Blade Runner 2049, and Barbie.

The Notebook is a romantic drama released in 2004, in which Ryan Gosling plays Noah Calhoun. 
T


##B3 Sparse Retrieval

so in this method it doesnt work with embeddings instead it works just with tokens of the words and look for exact words in the provided context.


In [29]:
!pip install rank_bm25

In [32]:
from rank_bm25 import BM25Okapi

chunk_tokens = [chunk.lower().split() for chunk in chunks]

bm25 = BM25Okapi(chunk_tokens)

In [50]:
def sparse_retrieve(query, bm25, k= 3):

  query_tokens = query.lower().split()

  scores = bm25.get_scores(query_tokens)

  top_scores = np.argsort(scores)[::-1][:k]

  top_results = []

  for i in top_scores:
    top_results.append((i, scores[i]))

  return top_results



In [51]:
query1 = "Am I ryan gosling ?"

result1 = sparse_retrieve(query1,bm25)

print("Query: ", query1)

for rank, (i, score) in enumerate(result1, start = 1):
  print(f"Rank {rank}")
  print(f"BM25 Score: {score:.4f}")
  print(f"Chunk: {chunks[i]}")

Query:  Am I ryan gosling ?
Rank 1
BM25 Score: 1.9719
Chunk: 
So first of all Ryan Gosling is literally me, Ryan Gosling is a Canadian actor known for taking on a wide variety of roles across different genres. 
His film career includes romantic dramas, crime thrillers, science fiction films, comedies, and musi
Rank 2
BM25 Score: 1.3644
Chunk:  in which Ryan Gosling plays Noah Calhoun. 
The story follows Noah and Allie, two people who fall deeply in love but face opposition from Allie's family. 
The movie became one of Gosling's most recognizable romantic roles and helped establish him as 
Rank 3
BM25 Score: 1.3644
Chunk: rs, science fiction films, comedies, and musicals. 
Some of his well-known movies include The Notebook, Drive, La La Land, Blade Runner 2049, and Barbie.

The Notebook is a romantic drama released in 2004, in which Ryan Gosling plays Noah Calhoun. 
T


In [44]:
query2 = "In which movie is Ryan Gosling paired with Margot Robbie ?"

result2 = sparse_retrieve(query2,bm25)

print("Query: ", query2)

for rank, (i, score) in enumerate(result2, start = 1):
  print(f"Rank {rank}")
  print(f"Similarity with the query: {score:.4f}")
  print(f"Chunk: {chunks[i]}")

Query:  In which movie is Ryan Gosling paired with Margot Robbie ?
Rank 1
Similarity with the query: 4.6681
Chunk: as released in 2023 and features Gosling as Ken opposite Margot Robbie as Barbie. 
The film follows Barbie as she leaves Barbieland and enters the real world, while Ken also begins to question his role and identity. 
Gosling's performance as Ken comb
Rank 2
Similarity with the query: 3.9390
Chunk:  in which Ryan Gosling plays Noah Calhoun. 
The story follows Noah and Allie, two people who fall deeply in love but face opposition from Allie's family. 
The movie became one of Gosling's most recognizable romantic roles and helped establish him as 
Rank 3
Similarity with the query: 3.7347
Chunk: rs, science fiction films, comedies, and musicals. 
Some of his well-known movies include The Notebook, Drive, La La Land, Blade Runner 2049, and Barbie.

The Notebook is a romantic drama released in 2004, in which Ryan Gosling plays Noah Calhoun. 
T


##Which is Better ?

In my opinion, although sparse retrieval method (B3) worked better with the above queries, when we consider a broader point of view then the dense retrieval methods will be better as it helps to identify similarity between different words with the same meaning which might help to understand complex relationships.